In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("./data/penguins.csv")
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


In [4]:
df.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

In [5]:
df = df.dropna() # drop rows with missing values (11 rows)

In [6]:
X = df.drop(columns="species")
y = df["species"]

In [7]:
le_island, le_sex, le = LabelEncoder(), LabelEncoder(), LabelEncoder()

X["island"] = le_island.fit_transform(X["island"]) # Biscoe, Dream, Torgersen -> 0, 1, 2
X["sex"] = le_sex.fit_transform(X["sex"]) # FEMALE, MALE -> 0, 1
y = le.fit_transform(y) # Adelie, Chinstrap, Gentoo -> 0, 1, 2
le.classes_

array(['Adelie', 'Chinstrap', 'Gentoo'], dtype=object)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.3, random_state=42, stratify=y)

In [9]:
scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train) # fit on TRAIN only
Xte = scaler.transform(X_test)

In [10]:
Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
Xte_t = torch.tensor(Xte, dtype=torch.float32)
ytr_t = torch.tensor(y_train, dtype=torch.long) # class labels must be long (int)

train_ds = TensorDataset(Xtr_t, ytr_t) # pairs (x, y) row by row
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True) # gives mini-batches of 32 rows

In [11]:
xb, yb = next(iter(train_loader)) # look at one batch
xb.shape, yb.shape

(torch.Size([32, 6]), torch.Size([32]))

In [12]:
class Model(nn.Module):
	def __init__(self, n_in, n_hidden, n_out):
		super().__init__()
		self.fc1 = nn.Linear(n_in, n_hidden)
		self.fc2 = nn.Linear(n_hidden, n_out)

	def forward(self, x):
		return self.fc2(torch.relu(self.fc1(x))) # raw scores (logits), no softmax

In [13]:
torch.manual_seed(42)
model = Model(n_in=Xtr.shape[1], n_hidden=20, n_out=len(le.classes_))
loss_fn = nn.CrossEntropyLoss() # softmax + cross-entropy in one
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 50
history = []

for epoch in range(epochs):
	model.train()
	total = 0
	for xb, yb in train_loader: # one mini-batch at a time
		y_hat = model(xb) # 1. forward
		loss = loss_fn(y_hat, yb) # 2. loss
		optimizer.zero_grad() # clear old gradients
		loss.backward() # 3. backward
		optimizer.step() # 4. update weights
		total += loss.item() * len(xb)
	history.append(total / len(train_ds)) # average loss of the epoch
	if epoch % 10 == 0:
		print(f"epoch {epoch:3d}  loss {history[-1]:.4f}")

epoch   0  loss 0.8406
epoch  10  loss 0.0244
epoch  20  loss 0.0093
epoch  30  loss 0.0055
epoch  40  loss 0.0057


In [14]:
model.eval()
with torch.no_grad(): # no gradients needed for predicting
	y_pred_train = model(Xtr_t).argmax(1).numpy() # highest score -> class
	y_pred_test = model(Xte_t).argmax(1).numpy()

print(f"Train  Acc: {accuracy_score(y_train, y_pred_train):.3f}")
print(f"Test   Acc: {accuracy_score(y_test, y_pred_test):.3f}")
print(classification_report(y_test, y_pred_test, target_names=le.classes_))

Train  Acc: 1.000
Test   Acc: 0.990
              precision    recall  f1-score   support

      Adelie       1.00      0.98      0.99        44
   Chinstrap       0.95      1.00      0.98        20
      Gentoo       1.00      1.00      1.00        36

    accuracy                           0.99       100
   macro avg       0.98      0.99      0.99       100
weighted avg       0.99      0.99      0.99       100



In [15]:
pd.DataFrame(confusion_matrix(y_test, y_pred_test), index=le.classes_, columns=le.classes_) # rows = actual, cols = predicted

,Adelie,Chinstrap,Gentoo
Adelie,43,1,0
Chinstrap,0,20,0
Gentoo,0,0,36


In [16]:
SAVE_CSV = False

submission_df = df.copy()
with torch.no_grad():
	pred = model(torch.tensor(scaler.transform(X), dtype=torch.float32)).argmax(1).numpy()
submission_df["species"] = le.inverse_transform(pred) # 0, 1, 2 -> text
if SAVE_CSV:
	submission_df.to_csv("submission.csv", index=False)

submission_df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,FEMALE
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE
